# Nash-Sutcliffe Efficiency (NSE)
In this tutorial, we will calculate the Nash-Sutcliffe Efficiency (NSE) coefficient using the function `scores.continuous.nse`.

NSE is a widely used metric in hydrology and other fields to evaluate the performance of a model by comparing its predictions to observed data.

## Definition

The Nash-Sutcliffe Efficiency (NSE) is defined as:

$$
\begin{align*}
\large{\text{NSE}} &= 1 - \large\frac{\large\sum_{i=1}^{n}(f_i - o_i)^2}{\large\sum_{i=1}^{n}(o_i - \bar{o})^2} \\ 
                     &= 1 - \large\frac{\text{MSE}}{\sigma_{o}^2} \\ 
\end{align*}
$$
<br>Where:
- $f_i$ is the forecast or predicted value
- $o_i$ is the observed value
- $\bar{o}$ is the mean of the observed values
- $n$ is the number of data points
- $\sigma_{\large{o}}^2$ is the un-weighted observation variance
- MSE is the mean-squared error, equivilent to the `scores` function: `scores.continuous.mse`


Scores also supports the use of `weights`. These can be used to scale both the individual forecast error in the numerator and the deviation from the observation mean in the denominator. This is also known as the weighted-NSE or wNSE (for an application of this, see: [Hundecha and Bárdossy (2004)](https://doi.org/10.1016/j.jhydrol.2004.01.002)).
<br>
$$
\begin{align*}
\large{\text{wNSE}} &= 1 - \large\frac{\large\sum_{i=1}^{n}w_i.(f_i - o_i)^2}{\large\sum_{i=1}^{n}w_i.(o_i - \bar{o})^2} \\ 
                      &= 1 - \large\frac{\text{MWSE}}{\sigma_{o_w}^2} \\ 
\end{align*}
$$

<br>Where:
- $\vec{w} = (w_1, w_2,..., w_n)$ are the weights for each index $i$
- The weights must be non-negative: $\forall i \ : \ 0 \leq w_i\ , \ \text{and there exists at least one}\ \  i \ : \ 0 \lt w_i$  (specifiable using `weights` option)
- $\sigma_{\large{o_w}}^2$ is the weighted obs variance
- $\small\text{MWSE}$ is the mean of the weighted square errors - again this is also equivilent to `scores.continuous.mse`, if using the weights argument

 > **caution:** $\small\text{MWSE}$ (scores) should not be confused with $\small\text{WMSE}$ (weighted mean square errror):
 > - $\small\text{WMSE}$ is a _"weighted mean"_ of _"unweighted"_ error terms.<br>
 > - On the other hand, $\small\text{MWSE}$ is a _"unweighted mean"_ of _"weighted"_ error terms.<br>
 > - The distinction is subtle, essentially: $\small\text{WMSE}$ treats 0 weights as data to *exclude*; additionally it requires that the *weights sum to 1*.
 > - Whereas $\small\text{MWSE}$ (what `scores` does), treats 0 weights as data that is *zero forced* and has no constraints on the upper bound of the *weights* (at the time of writing).
 > - Incidentally $\small\text{MWSE}$ is also what is needed to compute a "weighted" NSE ($\small\text{wNSE}$).

Further, while $i$ above is shown as a integer index from 0 to n, its definition can be abstracted to any _multi-index_ i.e.

$\vec{i} \in \{ \left(\ i_0,\ i_1,\ ...\ \right) :\  0\leq i_0 \lt N_0;\ 0\leq i_1 \lt N_1; \ ...\ \}$

If one wishes to accumulate the NSE scores over multiple indices, the `reduce_dims` and `preserve_dims` arguments can be used (similar to other scores).

## Interpretation

A perfect model has an NSE value of 1, while a model performing as poorly as the mean of the observed data has an NSE value of 0 or less.

In hydrological modeling, the Nash–Sutcliffe Efficiency (NSE) is essential for assessing model performance. An NSE of 1 indicates perfect prediction, while 0 suggests the model performs as well as predicting the mean of the data. Negative values imply the observed mean is a better predictor. NSE values closer to 1 denote superior predictive ability. In regression analyses, NSE parallels the coefficient of determination (R²), representing model fit on a scale from 0 to 1.

> **Note:**
> 
> The image (co-domain) of $R^2 : (x, x_\text{fit}) \rightarrow \text{score}$ does not necessarily need to be constrained to $[0, 1]$. It's just that a linear least squares optimisation makes it impossible for a fit to do worse than the mean.
>
> Analysis using NSE explicitly doesn't pose such restrictions. Its more akin to the signal to noise ratio in this sense, in fact
>
> $\text{SNR} \Large = \frac{E[{O^2}]}{E[\xi_\text{model}^2]} = \frac{\sigma_{o}^2}{\text{MSE}} = \frac{1}{1 - NSE}$
> 
> Where, $O$ is the observed process ("signal"), and $\xi_\text{model}$ is the error of the modelled predictions/forecasts/simulations ("noise")

## References
1. Nash, J. E., & Sutcliffe, J. V. (1970). River flow forecasting through conceptual models part I — A discussion of principles. Journal of Hydrology, 10(3), 282–290. https://doi.org/10.1016/0022-1694(70)90255-6 <br>
2. Hundecha, Y., & Bárdossy, A. (2004). Modeling of the effect of land use changes on the runoff generation of a river basin through parameter regionalization of a watershed model. Journal of Hydrology, 292(1–4), 281–295. https://doi.org/10.1016/j.jhydrol.2004.01.002 <br>


## Using the `nse` Function

Let's start by importing the `nse` function from our module and exploring its usage with different types of input data.

In [1]:
from scores.continuous import nse
import numpy as np
import xarray as xr
import pandas as pd

np.random.seed(0)  # set the seed to make notebook reproducible

### Example 1: Xarray DataArray

In [2]:
# build temperature dataarray - offset by 1:
temp_obs = np.linspace(start=0, stop=48, num=48, dtype=int, endpoint=False)
temp_fcst = np.linspace(start=1, stop=49, num=48, dtype=int, endpoint=False)
temp_obs = np.reshape(temp_obs, (2, 4, 2, 3))
temp_fcst = np.reshape(temp_fcst, (2, 4, 2, 3))

obs_xr = xr.DataArray(temp_obs, dims=['x', 'y', 't', 'l'])
fcst_xr = xr.DataArray(temp_fcst, dims=['x', 'y', 't', 'l'])
reduce_dims = ("t", "l")

In [3]:
nse_value = nse(fcst_xr, obs_xr, reduce_dims=reduce_dims)
nse_value

<xarray.DataArray 'NSE' (x: 2, y: 4)> Size: 64B
array([[0.65714286, 0.65714286, 0.65714286, 0.65714286],
       [0.65714286, 0.65714286, 0.65714286, 0.65714286]])
Dimensions without coordinates: x, y

In [ ]:
46/70

In [ ]:
temp_fcst

In [ ]:
temp_obs

In [ ]:
temp_obs.shape

### Reduce dimensions

In [ ]:
# Check with flattening
# Reshape the array to (a, b) by flattening the last two dimensions
temp_obs_2d = temp_obs.reshape(temp_obs.shape[0], temp_obs.shape[1],-1)
temp_fcst_2d = temp_fcst.reshape(temp_fcst.shape[0], temp_fcst.shape[1],-1)

In [ ]:
temp_obs_2d

In [ ]:
temp_fcst_2d

In [ ]:
obs_xr1 = xr.DataArray(temp_obs_2d, dims=['x', 'y', 't'])
fcst_xr1 = xr.DataArray(temp_fcst_2d, dims=['x', 'y', 't'])
nse(fcst_xr1, obs_xr1, reduce_dims="t")

In [ ]:
nse_value

In [ ]:
for i in range(obs_xr1.shape[0]):
    for j in range(obs_xr1.shape[1]):
        f = temp_fcst_2d[i,j,:]
        o = temp_obs_2d[i,j,:]
        mse = np.mean((f - o)**2)
        obs_var = np.var(o)
        print(f"mse: {mse}, obs_var: {obs_var}, nse: {1 - mse/obs_var}") 

In [ ]:
70/24

In [ ]:
46/70

### Compute for precip

In [ ]:
precip_obs = np.linspace(start=10, stop=58, num=48, dtype=int, endpoint=False)
precip_fcst = np.linspace(start=10, stop=58, num=48, dtype=int, endpoint=False)
precip_fcst[slice(0, None, 2)] += 2  # offset every even index
precip_obs = np.reshape(precip_obs, (2, 4, 2, 3))
precip_fcst = np.reshape(precip_fcst, (2, 4, 2, 3))

In [ ]:
precip_obs

In [ ]:
precip_fcst

In [ ]:
obs_xr = xr.DataArray(precip_obs, dims=['x', 'y', 't', 'l'])
fcst_xr = xr.DataArray(precip_fcst, dims=['x', 'y', 't', 'l'])
reduce_dims = ("t", "l")
nse_value = nse(fcst_xr, obs_xr, reduce_dims=reduce_dims)
nse_value

In [ ]:
22/70

In [ ]:
# Check with flattening
# Reshape the array to (a, b) by flattening the last two dimensions
precip_obs_2d = precip_obs.reshape(precip_obs.shape[0], precip_obs.shape[1],-1)
precip_fcst_2d = precip_fcst.reshape(precip_fcst.shape[0], precip_fcst.shape[1],-1)

In [ ]:
for i in range(precip_obs_2d.shape[0]):
    for j in range(precip_obs_2d.shape[1]):
        f = precip_fcst_2d[i,j,:]
        o = precip_obs_2d[i,j,:]
        mse = np.mean((f - o)**2)
        obs_var = np.var(o)
        print(f"mse: {mse}, obs_var: {obs_var}, nse: {1 - mse/obs_var}") 

In [ ]:
70/24

#### With weights

In [ ]:
temp_weights = np.array([[[[0, 3, 1]]], [[[0, 6, 3]]]])  # x,.,.,l: 2*1*1*3
precip_weights = np.array([[[[0, 1, 0], [2, 0, 2]]]])  # .,.,t,l: 1*1*2*3
temp_weights

In [ ]:
temp_weights.shape

In [ ]:
precip_weights

In [ ]:
precip_weights.shape

In [ ]:
expand_dims = {"x": 2, "y": 4, "t": 2, "l": 3}
ds_weights = xr.Dataset(
            {
                "temperature": xr.DataArray(
                    np.broadcast_to(temp_weights, expand_dims.values()),
                    dims=expand_dims.keys(),
                ),
                "precipitation": xr.DataArray(
                    np.broadcast_to(precip_weights, expand_dims.values()),
                    dims=expand_dims.keys(),
                ),
            }
        )
ds_weights

In [ ]:
ds_weights['temperature'].values.shape

In [ ]:
ds_weights['temperature'].values

In [ ]:
nse_value = nse(fcst_xr, obs_xr, reduce_dims=reduce_dims, weights=ds_weights['temperature'])
nse_value

In [ ]:
3/5

In [ ]:
ts1 = 3.0 / 5.0
ts2 = 114.0 / 186.0

In [ ]:
np.array([[ts1] * 4, [ts2] * 4])

In [ ]:
[1] * 4

In [ ]:
# Check with flattening
# Reshape the array to (a, b) by flattening the last two dimensions
temp_obs_2d = temp_obs.reshape(temp_obs.shape[0], temp_obs.shape[1],-1)
temp_fcst_2d = temp_fcst.reshape(temp_fcst.shape[0], temp_fcst.shape[1],-1)
w_2d= ds_weights['temperature'].values.reshape(ds_weights['temperature'].values.shape[0], ds_weights['temperature'].values.shape[1],-1)

In [ ]:
w

In [ ]:
for i in range(obs_xr1.shape[0]):
    for j in range(obs_xr1.shape[1]):
        f = temp_fcst_2d[i,j,:]
        o = temp_obs_2d[i,j,:]
        w = w_2d[i,j,:]
        
        mse = np.sum(w*(f - o)**2)
        obs_var = np.sum(w*(o - np.mean(o))**2)
        print(f"wmse: {mse}, wobs_var: {obs_var}, nse: {1 - mse/obs_var}") 

In [ ]:
186.0 / 24.0 == 46.5/6

In [ ]:
114/186

### Precipitation

In [ ]:
nse(fcst_xr, obs_xr, reduce_dims=reduce_dims, weights=ds_weights['precipitation'])

In [ ]:
ds_weights['precipitation']

In [ ]:
# Check with flattening
# Reshape the array to (a, b) by flattening the last two dimensions
precip_obs_2d = precip_obs.reshape(precip_obs.shape[0], precip_obs.shape[1],-1)
precip_fcst_2d = precip_fcst.reshape(precip_fcst.shape[0], precip_fcst.shape[1],-1)
w_2d= ds_weights['precipitation'].values.reshape(ds_weights['precipitation'].values.shape[0], ds_weights['precipitation'].values.shape[1],-1)

In [ ]:
for i in range(precip_obs_2d.shape[0]):
    for j in range(precip_obs_2d.shape[1]):
        f = precip_fcst_2d[i,j,:]
        o = precip_obs_2d[i,j,:]
        w = w_2d[i,j,:]
        
        mse = np.sum(w*(f - o)**2)
        obs_var = np.sum(w*(o - np.mean(o))**2)
        print(f"wmse: {mse}, wobs_var: {obs_var}, nse: {1 - mse/obs_var}") 

In [ ]:
61.0 / 24.0 == 15.25/6

### Check with R

In [ ]:
obs = xr.DataArray([2, 3, 4, 5, 6])
fcst = xr.DataArray([3, 4, 5, 6, 7])
w = obs
nse_val = nse(fcst, obs)
nse_val_w = nse(fcst, obs, weights=w)

In [ ]:
nse_val,nse_val_w

In [ ]:
sum([1.2/1.7,0.5/1.7])

### Check weighted MSE and skileanr weighted MSE 

In [ ]:
from scores.continuous import mse

In [ ]:
y_true = np.array([3, -0.5, 2, 7])
y_pred = np.array([2.5, 0.0, 2, 8])
weight = np.array([1,5,3,2])


In [ ]:
mse_val = mse(y_pred,y_true)
print(mse_val)

In [ ]:
mse_val_weight = mse(y_pred,y_true,weights=weight)
print(mse_val_weight)   
print(mse_val)

In [ ]:
nse_value = nse(fcst_xr, obs_xr)

In [ ]:
nse_value

In [3]:
nse(fcst_xr, obs_xr, preserve_dims='all')

DimensionError: 
    NSE: need at least one dimension to be reduced. Check that
    `preserve_dims` is not preserving all dimensions, OR check that
    `reduce_dims` is specified with at least one dimension.
    

### Check with missing values

In [2]:
fcst_xr = xr.DataArray([3, 4, 5, 6, 7])
obs_xr = xr.DataArray([2, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for Xarray DataArray:", nse_xr.values)

NSE for Xarray DataArray: 0.5


In [3]:
fcst_xr = xr.DataArray([np.nan, 4, 5, 6, 7])
obs_xr = xr.DataArray([2, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for with missing fcst:", nse_xr.values)

NSE for with missing fcst: 0.19999999999999996


In [4]:
fcst_xr = xr.DataArray([3, 4, 5, 6, 7])
obs_xr = xr.DataArray([np.nan, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for with missing obs:", nse_xr.values)

NSE for with missing obs: 0.19999999999999996


In [5]:
fcst_xr = xr.DataArray([np.nan, 4, 5, 6, 7])
obs_xr = xr.DataArray([np.nan, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for with both missing on same event:", nse_xr.values)

NSE for with both missing on same event: 0.19999999999999996


In [6]:
fcst_xr = xr.DataArray([4, 5, 6, 7])
obs_xr = xr.DataArray([3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("True value NSE for with both missing on same event:", nse_xr.values)

True value NSE for with both missing on same event: 0.19999999999999996


In [5]:
from scores.continuous import mse

In [6]:
fcst_xr = xr.DataArray([np.nan, 4, 5, 6, 7])
obs_xr = xr.DataArray([2, 3, 4, 5, 6])
mse_xr = mse(fcst_xr, obs_xr)
print("MSE for Xarray DataArray:", mse_xr)

MSE for Xarray DataArray: <xarray.DataArray ()> Size: 8B
array(1.)


In [7]:
fcst_xr = xr.DataArray([4, 5, 6, 7])
obs_xr = xr.DataArray([3, 4, 5, 6])
mse_xr = mse(fcst_xr, obs_xr)
print("MSE for Xarray DataArray:", mse_xr)

MSE for Xarray DataArray: <xarray.DataArray ()> Size: 8B
array(1.)


In [9]:
fcst_xr = xr.DataArray([np.nan, 4, 5, 6, 7])
obs_xr = xr.DataArray([np.nan, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for Xarray DataArray:", nse_xr)

NSE for Xarray DataArray: <xarray.DataArray 'NSE' ()> Size: 8B
array(0.2)


In [10]:
fcst_xr = xr.DataArray([np.nan, 4, 5, 6, 7])
obs_xr = xr.DataArray([2, 3, 4, 5, 6])
mse_xr = mse(fcst_xr, obs_xr)
print("MSE for Xarray DataArray:", mse_xr)

MSE for Xarray DataArray: <xarray.DataArray ()> Size: 8B
array(1.)


In [11]:
fcst_xr = xr.DataArray([np.nan, 4, 5, 6, 7])
obs_xr = xr.DataArray([2, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for Xarray DataArray:", nse_xr)

NSE for Xarray DataArray: <xarray.DataArray 'NSE' ()> Size: 8B
array(0.5)


In [12]:
fcst_xr = xr.DataArray([3, 4, 5, 6, 7])
obs_xr = xr.DataArray([np.nan, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for Xarray DataArray:", nse_xr)

NSE for Xarray DataArray: <xarray.DataArray 'NSE' ()> Size: 8B
array(0.2)


In [13]:
fcst_xr = xr.DataArray([np.nan, 4, 5, 6, 7])
obs_xr = xr.DataArray([np.nan, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for Xarray DataArray:", nse_xr)

NSE for Xarray DataArray: <xarray.DataArray 'NSE' ()> Size: 8B
array(0.2)


In [14]:
fcst_xr = xr.DataArray([np.nan, 4, 5, 6, 7])
obs_xr = xr.DataArray([2, 3, 4, 5, 6])
nse_xr = nse(fcst_xr, obs_xr)
print("NSE for Xarray DataArray:", nse_xr)

NSE for Xarray DataArray: <xarray.DataArray 'NSE' ()> Size: 8B
array(0.5)
